# Final Benchmark Comparison for the Minimal Black--Scholes Hedging Task

This notebook implements the **Priority 1** final benchmark comparison.

It evaluates the selected neural hedge against the following strategies:

1. **No hedge**
2. **Black--Scholes delta hedge** sampled on the discrete rebalancing grid
3. **Discrete-time MSE-optimal hedge**
4. **Polynomial hedge baseline**
5. **Final selected neural hedge**

The goal is to produce a clean final table with:

\[
\text{Premium},\quad
\mathbb{E}[HE],\quad
\operatorname{Std}(HE),\quad
\operatorname{RMSE},\quad
q_{1\%}(HE),\quad
q_{5\%}(HE),\quad
\operatorname{CVaR}_{95},\quad
\operatorname{CVaR}_{99}.
\]

The sign convention is:

\[
HE = V_T - \Phi(S_T),
\]

so **negative hedge error means a shortfall/loss for the option seller**. For tail-risk metrics we define loss as:

\[
L = -HE.
\]

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

from dataclasses import dataclass, asdict
from pathlib import Path
import json
import time
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

@dataclass
class Config:
    # Black--Scholes minimal-task parameters
    S0: float = 1.0
    K: float = 0.9
    T: float = 0.5
    r: float = 0.0
    sigma: float = 0.4
    N: int = 125

    # Monte Carlo path counts
    n_train: int = 100_000
    n_val: int = 30_000
    n_test: int = 100_000

    # Neural network architecture: selected final shared normalized Markov MLP
    width: int = 64
    depth: int = 3
    hidden_activation: str = "tanh"
    output_activation: str = "sigmoid"

    # Training
    batch_size: int = 4096
    learning_rate: float = 1e-3
    max_epochs: int = 250
    patience: int = 20

    # Polynomial hedge baseline
    polynomial_degree: int = 2
    poly_fit_rows: int = 500_000

    # Reproducibility / outputs
    seed: int = 123
    output_dir: str = "final_benchmark_outputs"

    # Set to True for a quick smoke test before the full run
    quick_run: bool = False

cfg = Config()

if cfg.quick_run:
    cfg.n_train = 10_000
    cfg.n_val = 5_000
    cfg.n_test = 10_000
    cfg.max_epochs = 5
    cfg.patience = 3
    cfg.poly_fit_rows = 50_000

out_dir = Path(cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)

np.random.seed(cfg.seed)

print(json.dumps(asdict(cfg), indent=2))

# This notebook uses the self-financing wealth convention
# V_T = premium + sum delta_n (S_{n+1} - S_n).
# That convention is directly valid for r=0, which is the minimal-task setup.
if abs(cfg.r) > 1e-12:
    print("WARNING: This notebook is written for the r=0 minimal task. "
          "If r != 0, discounting / cash-account accrual should be added carefully.")

In [ ]:
# ============================================================
# 2. Black--Scholes utilities and GBM simulation
# ============================================================

try:
    from scipy.special import ndtr
except Exception as e:
    raise ImportError("This notebook requires scipy.special.ndtr for the normal CDF.") from e


def norm_cdf(x):
    return ndtr(x)


def bs_call_price(S, tau, cfg=cfg):
    """
    Black--Scholes European call price under risk-neutral dynamics.
    Handles scalar or array S and tau.
    """
    S = np.asarray(S, dtype=np.float64)
    tau = np.asarray(tau, dtype=np.float64)

    intrinsic = np.maximum(S - cfg.K, 0.0)
    tau_safe = np.maximum(tau, 1e-12)

    d1 = (np.log(S / cfg.K) + (cfg.r + 0.5 * cfg.sigma**2) * tau_safe) / (cfg.sigma * np.sqrt(tau_safe))
    d2 = d1 - cfg.sigma * np.sqrt(tau_safe)

    price = S * norm_cdf(d1) - cfg.K * np.exp(-cfg.r * tau_safe) * norm_cdf(d2)
    return np.where(tau <= 0, intrinsic, price)


def bs_call_delta(S, tau, cfg=cfg):
    """
    Black--Scholes European call delta.
    Handles scalar or array S and tau.
    """
    S = np.asarray(S, dtype=np.float64)
    tau = np.asarray(tau, dtype=np.float64)

    tau_safe = np.maximum(tau, 1e-12)
    d1 = (np.log(S / cfg.K) + (cfg.r + 0.5 * cfg.sigma**2) * tau_safe) / (cfg.sigma * np.sqrt(tau_safe))
    delta = norm_cdf(d1)

    # At expiry, the call delta is the payoff derivative away from the strike.
    expiry_delta = (S > cfg.K).astype(np.float64)
    return np.where(tau <= 0, expiry_delta, delta)


def simulate_gbm_paths(n_paths, cfg=cfg, seed=None):
    """
    Simulate risk-neutral GBM paths:
        dS_t = r S_t dt + sigma S_t dW_t

    Returns an array of shape (n_paths, N+1), including S_0.
    """
    rng = np.random.default_rng(seed)
    dt = cfg.T / cfg.N
    Z = rng.standard_normal(size=(n_paths, cfg.N)).astype(np.float32)

    increments = (cfg.r - 0.5 * cfg.sigma**2) * dt + cfg.sigma * np.sqrt(dt) * Z
    log_paths = np.cumsum(increments, axis=1)

    paths = np.empty((n_paths, cfg.N + 1), dtype=np.float32)
    paths[:, 0] = cfg.S0
    paths[:, 1:] = cfg.S0 * np.exp(log_paths)
    return paths


def call_payoff(S_T, cfg=cfg):
    return np.maximum(S_T - cfg.K, 0.0)


dt = cfg.T / cfg.N
t_grid = np.arange(cfg.N) * dt
tau_grid = cfg.T - t_grid
tau_norm_grid = tau_grid / cfg.T

bs_price_0 = float(bs_call_price(cfg.S0, cfg.T, cfg))
print(f"Black--Scholes initial price: {bs_price_0:.8f}")

In [ ]:
# ============================================================
# 3. Generate independent train / validation / test paths
# ============================================================

start = time.time()

paths_train = simulate_gbm_paths(cfg.n_train, cfg, seed=cfg.seed + 1)
paths_val = simulate_gbm_paths(cfg.n_val, cfg, seed=cfg.seed + 2)
paths_test = simulate_gbm_paths(cfg.n_test, cfg, seed=cfg.seed + 3)

payoff_train = call_payoff(paths_train[:, -1], cfg)
payoff_val = call_payoff(paths_val[:, -1], cfg)
payoff_test = call_payoff(paths_test[:, -1], cfg)

print(f"Generated paths in {time.time() - start:.1f}s")
print("Train paths:", paths_train.shape)
print("Val paths:  ", paths_val.shape)
print("Test paths: ", paths_test.shape)

# Basic simulation sanity checks
mc_call = np.mean(payoff_test) * np.exp(-cfg.r * cfg.T)
mean_ST = np.mean(paths_test[:, -1])
theory_mean_ST = cfg.S0 * np.exp(cfg.r * cfg.T)
var_log_return = np.var(np.log(paths_test[:, -1] / cfg.S0))
theory_var_log_return = cfg.sigma**2 * cfg.T

print("\nSimulation checks:")
print(f"MC call price estimate: {mc_call:.8f}")
print(f"BS call price:          {bs_price_0:.8f}")
print(f"E[S_T] MC:              {mean_ST:.8f}")
print(f"E[S_T] theory:          {theory_mean_ST:.8f}")
print(f"Var(log S_T/S_0) MC:    {var_log_return:.8f}")
print(f"Var(log S_T/S_0) theory:{theory_var_log_return:.8f}")

In [ ]:
# ============================================================
# 4. Hedge-error and metric utilities
# ============================================================

def hedge_error(paths, deltas, premium, cfg=cfg):
    """
    Terminal hedging error:
        HE = V_T - payoff
           = premium + sum delta_n (S_{n+1} - S_n) - payoff(S_T)

    paths:  shape (M, N+1)
    deltas: shape (M, N)
    """
    dS = paths[:, 1:] - paths[:, :-1]
    terminal_wealth = premium + np.sum(deltas * dS, axis=1)
    payoff = call_payoff(paths[:, -1], cfg)
    return terminal_wealth - payoff


def cvar_loss_from_he(he, alpha=0.95):
    """
    CVaR_alpha of seller loss L = -HE.
    """
    loss = -np.asarray(he)
    var_alpha = np.quantile(loss, alpha)
    tail = loss[loss >= var_alpha]
    return float(np.mean(tail))


def strategy_metrics(name, he, premium, runtime=None):
    """
    Returns a dictionary of final benchmark metrics.
    """
    he = np.asarray(he)
    loss = -he
    out = {
        "Strategy": name,
        "Premium": float(premium),
        "Mean HE": float(np.mean(he)),
        "Std HE": float(np.std(he, ddof=1)),
        "RMSE": float(np.sqrt(np.mean(he**2))),
        "HE q01": float(np.quantile(he, 0.01)),
        "HE q05": float(np.quantile(he, 0.05)),
        "Loss VaR95": float(np.quantile(loss, 0.95)),
        "Loss CVaR95": cvar_loss_from_he(he, 0.95),
        "Loss VaR99": float(np.quantile(loss, 0.99)),
        "Loss CVaR99": cvar_loss_from_he(he, 0.99),
    }
    if runtime is not None:
        out["Runtime (s)"] = float(runtime)
    return out

In [ ]:
# ============================================================
# 5. Analytic benchmark hedges
# ============================================================

def bs_delta_hedge(paths, cfg=cfg):
    """
    Black--Scholes delta sampled at the discrete rebalancing dates.
    """
    S = paths[:, :-1].astype(np.float64)
    tau = tau_grid[None, :]
    deltas = bs_call_delta(S, tau, cfg)
    return deltas.astype(np.float32)


def dt_mse_optimal_delta_hedge(paths, cfg=cfg):
    """
    Discrete-time MSE-optimal hedge for the r=0 Black--Scholes minimal task.

    For stock martingale S and payoff H = (S_T-K)^+, the optimal predictable
    coefficient on each increment is

        delta_n^DT = E[H (S_{n+1}-S_n) | S_n] / E[(S_{n+1}-S_n)^2 | S_n].

    In the Black--Scholes lognormal model this has a closed-form expression.

    This benchmark is the fairest comparison for the neural hedge because it
    matches the finite rebalancing grid and MSE objective.
    """
    if abs(cfg.r) > 1e-12:
        raise ValueError("This closed-form DT benchmark is implemented for r=0.")

    h = cfg.T / cfg.N
    S = paths[:, :-1].astype(np.float64)
    tau = tau_grid[None, :].astype(np.float64)
    tau_safe = np.maximum(tau, 1e-12)

    sqrt_tau = np.sqrt(tau_safe)
    d1 = (np.log(S / cfg.K) + 0.5 * cfg.sigma**2 * tau_safe) / (cfg.sigma * sqrt_tau)
    d2 = d1 - cfg.sigma * sqrt_tau

    C = bs_call_price(S, tau, cfg)

    shift = cfg.sigma * h / sqrt_tau
    numerator = (
        S**2 * np.exp(cfg.sigma**2 * h) * norm_cdf(d1 + shift)
        - cfg.K * S * norm_cdf(d2 + shift)
        - S * C
    )
    denominator = S**2 * (np.exp(cfg.sigma**2 * h) - 1.0)

    delta = numerator / denominator
    return delta.astype(np.float32)


# Compute benchmark deltas on test paths
start = time.time()
deltas_no = np.zeros((cfg.n_test, cfg.N), dtype=np.float32)
he_no = hedge_error(paths_test, deltas_no, bs_price_0, cfg)
time_no = time.time() - start

start = time.time()
deltas_bs = bs_delta_hedge(paths_test, cfg)
he_bs = hedge_error(paths_test, deltas_bs, bs_price_0, cfg)
time_bs = time.time() - start

start = time.time()
deltas_dt = dt_mse_optimal_delta_hedge(paths_test, cfg)
he_dt = hedge_error(paths_test, deltas_dt, bs_price_0, cfg)
time_dt = time.time() - start

benchmark_rows = [
    strategy_metrics("No hedge", he_no, bs_price_0, runtime=time_no),
    strategy_metrics("Black--Scholes delta", he_bs, bs_price_0, runtime=time_bs),
    strategy_metrics("Discrete-time MSE-optimal", he_dt, bs_price_0, runtime=time_dt),
]

pd.DataFrame(benchmark_rows)

In [ ]:
# ============================================================
# 6. Polynomial hedge baseline
# ============================================================

def polynomial_design(x1, x2, degree=2):
    """
    Polynomial features in two variables x1 and x2 up to total degree.
    Includes an intercept.
    """
    cols = [np.ones_like(x1)]
    for total_degree in range(1, degree + 1):
        for power_x1 in range(total_degree + 1):
            power_x2 = total_degree - power_x1
            cols.append((x1 ** power_x1) * (x2 ** power_x2))
    return np.column_stack(cols)


def fit_polynomial_to_dt_delta(paths, cfg=cfg, degree=2, n_rows=500_000, seed=123):
    """
    Fits a simple polynomial approximation to the analytic DT-optimal delta.
    This is a non-neural function-approximation benchmark.

    To avoid building a huge design matrix, we sample random (path,time) pairs.
    """
    rng = np.random.default_rng(seed)
    M = paths.shape[0]
    N = cfg.N

    # Compute DT labels on training paths.
    dt_labels = dt_mse_optimal_delta_hedge(paths, cfg)

    n_rows = min(n_rows, M * N)
    path_idx = rng.integers(0, M, size=n_rows)
    time_idx = rng.integers(0, N, size=n_rows)

    S_sample = paths[path_idx, time_idx].astype(np.float64)
    x1 = np.log(S_sample / cfg.K)
    x2 = tau_norm_grid[time_idx].astype(np.float64)
    y = dt_labels[path_idx, time_idx].astype(np.float64)

    X = polynomial_design(x1, x2, degree)
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    return coef


def predict_polynomial_delta(paths, coef, cfg=cfg, degree=2, path_batch=10_000):
    """
    Predict polynomial hedge deltas in path batches to avoid large memory spikes.
    """
    M = paths.shape[0]
    out = np.empty((M, cfg.N), dtype=np.float32)

    for start in range(0, M, path_batch):
        end = min(start + path_batch, M)
        S = paths[start:end, :-1].astype(np.float64)
        x1 = np.log(S / cfg.K).reshape(-1)
        x2 = np.broadcast_to(tau_norm_grid[None, :], S.shape).reshape(-1)

        X = polynomial_design(x1, x2, degree)
        pred = X @ coef

        # For a European call hedge, clipping to [0,1] is economically natural.
        pred = np.clip(pred, 0.0, 1.0)
        out[start:end, :] = pred.reshape(end - start, cfg.N).astype(np.float32)

    return out


start = time.time()
poly_coef = fit_polynomial_to_dt_delta(
    paths_train,
    cfg=cfg,
    degree=cfg.polynomial_degree,
    n_rows=cfg.poly_fit_rows,
    seed=cfg.seed + 10,
)
deltas_poly = predict_polynomial_delta(paths_test, poly_coef, cfg=cfg, degree=cfg.polynomial_degree)
he_poly = hedge_error(paths_test, deltas_poly, bs_price_0, cfg)
time_poly = time.time() - start

benchmark_rows.append(strategy_metrics(f"Polynomial degree {cfg.polynomial_degree}", he_poly, bs_price_0, runtime=time_poly))
pd.DataFrame(benchmark_rows)

In [ ]:
# ============================================================
# 7. Neural hedge model
# ============================================================

try:
    import tensorflow as tf
except Exception as e:
    raise ImportError(
        "TensorFlow is required for the neural hedge section. "
        "Install tensorflow or run this section in the same environment as the tuning notebook."
    ) from e

tf.keras.utils.set_random_seed(cfg.seed)


class SharedMarkovHedge(tf.keras.Model):
    """
    Shared Markov MLP:
        delta_n = f_theta(log(S_n/K), tau_n/T)

    The same neural network is reused at every hedging date.
    """
    def __init__(self, cfg, initial_premium):
        super().__init__()
        self.cfg = cfg
        self.premium = self.add_weight(
            name="premium",
            shape=(),
            initializer=tf.keras.initializers.Constant(initial_premium),
            trainable=True,
            dtype=tf.float32,
        )

        layers = [tf.keras.layers.InputLayer(shape=(2,))]
        for _ in range(cfg.depth):
            layers.append(tf.keras.layers.Dense(cfg.width, activation=cfg.hidden_activation))
        layers.append(tf.keras.layers.Dense(1, activation=cfg.output_activation))
        self.net = tf.keras.Sequential(layers)

        self.tau_norm_tf = tf.constant(tau_norm_grid.astype(np.float32), dtype=tf.float32)

    def make_features(self, paths):
        S = paths[:, :-1]
        log_m = tf.math.log(S / tf.constant(self.cfg.K, dtype=tf.float32))
        tau_feat = tf.broadcast_to(self.tau_norm_tf[None, :], tf.shape(S))
        return tf.stack([log_m, tau_feat], axis=-1)

    def call(self, paths, training=False):
        features = self.make_features(paths)
        batch_size = tf.shape(paths)[0]

        flat_features = tf.reshape(features, (-1, 2))
        flat_delta = self.net(flat_features, training=training)
        deltas = tf.reshape(flat_delta[:, 0], (batch_size, self.cfg.N))

        dS = paths[:, 1:] - paths[:, :-1]
        payoff = tf.nn.relu(paths[:, -1] - tf.constant(self.cfg.K, dtype=tf.float32))

        terminal_wealth = self.premium + tf.reduce_sum(deltas * dS, axis=1)
        he = terminal_wealth - payoff
        return he, deltas


def train_neural_hedge(paths_train, paths_val, cfg=cfg, initial_premium=bs_price_0):
    model = SharedMarkovHedge(cfg, initial_premium)
    optimizer = tf.keras.optimizers.Adam(learning_rate=cfg.learning_rate)

    train_ds = (
        tf.data.Dataset.from_tensor_slices(paths_train.astype(np.float32))
        .shuffle(buffer_size=min(len(paths_train), 50_000), seed=cfg.seed, reshuffle_each_iteration=True)
        .batch(cfg.batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )

    val_ds = (
        tf.data.Dataset.from_tensor_slices(paths_val.astype(np.float32))
        .batch(cfg.batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )

    @tf.function
    def train_step(batch):
        with tf.GradientTape() as tape:
            he, _ = model(batch, training=True)
            loss = tf.reduce_mean(tf.square(he))
        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        return loss

    @tf.function
    def eval_step(batch):
        he, _ = model(batch, training=False)
        return tf.reduce_sum(tf.square(he)), tf.shape(he)[0]

    best_val = np.inf
    best_weights = None
    wait = 0
    history = []

    start_time = time.time()

    for epoch in range(1, cfg.max_epochs + 1):
        train_losses = []
        for batch in train_ds:
            loss = train_step(batch)
            train_losses.append(float(loss.numpy()))

        val_sse = 0.0
        val_n = 0
        for batch in val_ds:
            sse, n = eval_step(batch)
            val_sse += float(sse.numpy())
            val_n += int(n.numpy())

        train_loss = float(np.mean(train_losses))
        val_loss = val_sse / val_n

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "premium": float(model.premium.numpy()),
            "elapsed_s": time.time() - start_time,
        })

        improved = val_loss < best_val - 1e-10
        if improved:
            best_val = val_loss
            best_weights = model.get_weights()
            wait = 0
        else:
            wait += 1

        if epoch == 1 or epoch % 10 == 0 or improved:
            print(
                f"Epoch {epoch:03d} | train {train_loss:.8e} | val {val_loss:.8e} "
                f"| premium {float(model.premium.numpy()):.8f} | wait {wait}"
            )

        if wait >= cfg.patience:
            print(f"Early stopping at epoch {epoch}. Best val loss = {best_val:.8e}")
            break

    if best_weights is not None:
        model.set_weights(best_weights)

    history_df = pd.DataFrame(history)
    return model, history_df, time.time() - start_time


def predict_neural_hedge(model, paths, cfg=cfg):
    ds = tf.data.Dataset.from_tensor_slices(paths.astype(np.float32)).batch(cfg.batch_size)

    all_he = []
    all_deltas = []
    for batch in ds:
        he, deltas = model(batch, training=False)
        all_he.append(he.numpy())
        all_deltas.append(deltas.numpy())

    return np.concatenate(all_he), np.concatenate(all_deltas, axis=0)


start = time.time()
nn_model, nn_history, train_runtime = train_neural_hedge(paths_train, paths_val, cfg, bs_price_0)
he_nn, deltas_nn = predict_neural_hedge(nn_model, paths_test, cfg)
time_nn_total = time.time() - start

nn_premium = float(nn_model.premium.numpy())

benchmark_rows.append(strategy_metrics("Neural hedge", he_nn, nn_premium, runtime=time_nn_total))

nn_history.to_csv(out_dir / "nn_training_history.csv", index=False)
pd.DataFrame(benchmark_rows)

In [ ]:
# ============================================================
# 8. Final benchmark table
# ============================================================

results_df = pd.DataFrame(benchmark_rows)

# Make display a bit cleaner
display_cols = [
    "Strategy",
    "Premium",
    "Mean HE",
    "Std HE",
    "RMSE",
    "HE q01",
    "HE q05",
    "Loss VaR95",
    "Loss CVaR95",
    "Loss VaR99",
    "Loss CVaR99",
    "Runtime (s)",
]

results_df = results_df[display_cols]
results_df.to_csv(out_dir / "final_benchmark_metrics.csv", index=False)
results_df.to_latex(out_dir / "final_benchmark_metrics.tex", index=False, float_format="%.6g")

results_df

In [ ]:
# ============================================================
# 9. Plots: RMSE bar chart and hedge-error distributions
# ============================================================

# RMSE bar chart
plt.figure(figsize=(10, 5))
plt.bar(results_df["Strategy"], results_df["RMSE"])
plt.ylabel("RMSE of hedge error")
plt.title("Final benchmark comparison: hedge-error RMSE")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(out_dir / "benchmark_rmse_bar.png", dpi=200)
plt.show()

# Hedge error histograms
he_by_strategy = {
    "No hedge": he_no,
    "Black--Scholes delta": he_bs,
    "Discrete-time MSE-optimal": he_dt,
    f"Polynomial degree {cfg.polynomial_degree}": he_poly,
    "Neural hedge": he_nn,
}

plt.figure(figsize=(10, 6))
for name, he in he_by_strategy.items():
    # Clip plotting range for readability.
    lo, hi = np.quantile(he, [0.005, 0.995])
    plt.hist(
        np.clip(he, lo, hi),
        bins=80,
        density=True,
        histtype="step",
        linewidth=1.5,
        label=name,
    )

plt.axvline(0.0, linestyle="--", linewidth=1)
plt.xlabel("Hedge error HE = terminal wealth - payoff")
plt.ylabel("Density")
plt.title("Final benchmark comparison: hedge-error distributions")
plt.legend()
plt.tight_layout()
plt.savefig(out_dir / "benchmark_hedge_error_histograms.png", dpi=200)
plt.show()

print(f"Saved outputs to: {out_dir.resolve()}")

In [ ]:
# ============================================================
# 10. Optional: inspect average delta shapes by moneyness bucket
# ============================================================

# This lightweight diagnostic checks whether the learned deltas are economically sensible
# without producing a full 3D delta surface.

def binned_delta_summary(paths, deltas, name, n_bins=30):
    S = paths[:, :-1]
    moneyness = (S / cfg.K).reshape(-1)
    delta_flat = deltas.reshape(-1)

    bins = np.linspace(np.quantile(moneyness, 0.01), np.quantile(moneyness, 0.99), n_bins + 1)
    centers = 0.5 * (bins[:-1] + bins[1:])

    means = []
    for i in range(n_bins):
        mask = (moneyness >= bins[i]) & (moneyness < bins[i + 1])
        means.append(np.mean(delta_flat[mask]) if np.any(mask) else np.nan)

    return pd.DataFrame({"S_over_K": centers, "mean_delta": means, "strategy": name})


delta_summary = pd.concat([
    binned_delta_summary(paths_test, deltas_bs, "Black--Scholes delta"),
    binned_delta_summary(paths_test, deltas_dt, "Discrete-time MSE-optimal"),
    binned_delta_summary(paths_test, deltas_nn, "Neural hedge"),
], ignore_index=True)

plt.figure(figsize=(8, 5))
for name, group in delta_summary.groupby("strategy"):
    plt.plot(group["S_over_K"], group["mean_delta"], label=name)

plt.xlabel("Moneyness S/K")
plt.ylabel("Average hedge ratio")
plt.title("Average hedge ratio by moneyness")
plt.legend()
plt.tight_layout()
plt.savefig(out_dir / "average_delta_by_moneyness.png", dpi=200)
plt.show()

delta_summary.to_csv(out_dir / "average_delta_by_moneyness.csv", index=False)

## Report wording template

After running this notebook, the benchmark table can be described in the report using wording like:

> The neural hedge was compared against an unhedged short-call position, the standard Black--Scholes delta hedge sampled on the discrete rebalancing grid, a discrete-time MSE-optimal hedge, and a polynomial approximation to the discrete-time optimal hedge. The discrete-time MSE-optimal hedge is the fairest analytic benchmark because it matches the finite rebalancing grid and mean-square objective used to train the neural network. The neural network is not expected to materially outperform this benchmark under the correctly specified Black--Scholes model; rather, strong performance is indicated by matching its hedging-error distribution and producing a learned premium close to the analytic Black--Scholes price.

Remember the sign convention:

\[
HE = V_T - \Phi(S_T).
\]

Thus, more negative lower-tail hedge errors correspond to larger seller shortfalls.